In [0]:
# check_and_trigger_gold
# =======================
# Checks pipeline_run_log table to see if both CDR and Events
# pipelines have completed for today — no time window needed!
# Checks is_processed = 0 to avoid triggering gold multiple times
# If both complete and not yet processed → triggers gold job via API

import urllib.request
import json
from datetime import datetime

# ── CONFIG ────────────────────────────────────────────────────────────────────
DATABRICKS_HOST  = "<DATABRICKS_HOSTNAME>"
DATABRICKS_TOKEN = "<DATABRICKS_PAT>"
GOLD_JOB_ID      = "391939335730129"
# ─────────────────────────────────────────────────────────────────────────────

today = datetime.now().strftime("%Y-%m-%d")

print(f"{'='*50}")
print(f"Checking pipeline_run_log for: {today}")
print(f"{'='*50}")

# ── Check pipeline_run_log for today ─────────────────────────────────────────
log = spark.sql(f"""
    SELECT   pipeline,
             MAX(is_processed) as is_processed,
             MAX(logged_at)    as last_logged
    FROM     telecom.bronze.pipeline_run_log
    WHERE    run_date = '{today}'
    GROUP BY pipeline
""").collect()

completed = {row["pipeline"]: row for row in log}
print(f"Pipelines logged today: {list(completed.keys())}")

# ── Check CDR ─────────────────────────────────────────────────────────────────
if "cdr" not in completed:
    print("\nCDR pipeline not logged yet — exiting cleanly")
    dbutils.notebook.exit("WAITING_FOR_CDR")

# ── Check Events ──────────────────────────────────────────────────────────────
if "events" not in completed:
    print("\nEvents pipeline not logged yet — exiting cleanly")
    dbutils.notebook.exit("WAITING_FOR_EVENTS")

# ── Check if already processed into gold ─────────────────────────────────────
if completed["cdr"]["is_processed"] == 1:
    print(f"\nAlready processed into gold for {today} — skipping!")
    dbutils.notebook.exit("ALREADY_PROCESSED")

# ── Both ready + not yet processed — trigger gold ─────────────────────────────
cdr_logged    = completed["cdr"]["last_logged"]
events_logged = completed["events"]["last_logged"]

print(f"\nBoth pipelines ready for {today}!")
print(f"  CDR logged at:    {cdr_logged}")
print(f"  Events logged at: {events_logged}")
print(f"  is_processed:     0 → triggering gold!")

url     = f"https://{DATABRICKS_HOST}/api/2.1/jobs/run-now"
headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type":  "application/json"
}
payload  = json.dumps({"job_id": int(GOLD_JOB_ID)}).encode()
req      = urllib.request.Request(url, data=payload, headers=headers)

try:
    response = urllib.request.urlopen(req)
    result   = json.loads(response.read())
    run_id   = result.get("run_id")
    print(f"\nGold job triggered successfully!")
    print(f"  run_id: {run_id}")
    print(f"  job_id: {GOLD_JOB_ID}")
    dbutils.notebook.exit(f"GOLD_TRIGGERED run_id={run_id}")
except Exception as e:
    print(f"Failed to trigger gold job: {e}")
    raise